# Task 5 — Feature Engineering & Feature Selection

**Project:** Quantum Optimization for Formula 1 Race Strategy

This notebook walks through Task 5: turning Task 4's cleaned per-lap dataset into the
model-ready feature matrix that Task 6 (classical ML), Task 7 (deep learning) and Task 8 (XAI)
all read.

**The code lives in [`app/intelligence/features/build.py`](../../app/intelligence/features/build.py),
not in this notebook.** That way the same functions run here, in
`scripts/build_features.py`, and as a stage of `scripts/build_all.py` — so the committed
feature matrix can always be rebuilt from raw data, which was not true while this logic
existed only in notebook cells.

| | |
|---|---|
| Input | `data/processed/fastf1_laps_clean.csv` (Task 4) |
| Output | `data/processed/f1_features_selected.csv` — identifiers + selected features + targets |
| Output | `data/processed/feature_metadata.json` — feature lists, provenance, scaling/splitting contract |

Dependencies: `pandas`, `numpy`, `scikit-learn`. Variance-inflation factors are computed from
`sklearn.LinearRegression`, so `statsmodels` is not needed.

## 1 · Setup

Run the whole stage once. Everything shown below is read back from that single run, so the
notebook cannot drift from what the pipeline actually produced.

In [1]:
import json
import sys
from pathlib import Path

import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "app" / "intelligence").is_dir())
sys.path.insert(0, str(ROOT))

from app.intelligence.features import build  # noqa: E402  (needs sys.path set above)

pd.set_option("display.width", 160, "display.max_columns", 60)

result = build.build()          # writes the two contract files
laps, F = result["laps"], result["features_all"]
meta = result["metadata"]

print(f"rows x columns : {result['export'].shape[0]} x {result['export'].shape[1]}")
print(f"features built : {F.shape[1]}  ->  selected {len(meta['selected_features']['union_exported'])}")
print(f"random_state   : {build.RANDOM_STATE}  (every stochastic step is seeded)")

/Users/anjalimore/Desktop/projects/CIL/.venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/Users/anjalimore/Desktop/projects/CIL/.venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1429: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(


rows x columns : 995 x 53
features built : 50  ->  selected 45
random_state   : 42  (every stochastic step is seeded)


## 2 · The cleaned dataset

Task 4's output, as a driver-by-lap panel. `build.load_clean_laps` sorts it and asserts the
frame is complete and that `Driver`/`LapNumber` is unique.

In [2]:
print(f"drivers   : {laps['Driver'].nunique()}")
print(f"race laps : {int(laps['LapNumber'].min())}-{int(laps['LapNumber'].max())} (after the warm-up trim)")
print(f"compounds : {sorted(laps['Compound'].unique())}")
laps.head()

drivers   : 20
race laps : 4-57 (after the warm-up trim)
compounds : ['HARD', 'MEDIUM', 'SOFT']


,Driver,Team,LapNumber,LapTime,Stint,Compound,TyreLife,FreshTyre,Sector1Time,Sector2Time,Sector3Time,SpeedFL,SpeedST,TrackStatus,AirTemp,TrackTemp,Humidity,Rainfall,WindSpeed,IsPersonalBest,target_laptime,target_pit_next_lap,target_laptime_fuel_corrected,_field_median,_gap_to_field
0,ALB,WILLIAMS,4.0,99.761,1.0,SOFT,4.0,True,31.226,43.808,24.727,287.5,324.0,1,27.3,31.2,22.0,False,0.4,True,99.761,0,99.944979,99.7920,-0.0310
1,ALB,WILLIAMS,5.0,100.606,1.0,SOFT,5.0,True,31.480,44.210,24.916,284.0,321.0,1,27.2,31.0,22.0,False,0.9,False,100.606,0,100.851305,100.6180,-0.0120
2,ALB,WILLIAMS,6.0,100.744,1.0,SOFT,6.0,True,31.764,44.077,24.903,283.0,312.0,1,27.1,31.0,22.0,False,0.6,False,100.744,0,101.050632,100.5375,0.2065
3,ALB,WILLIAMS,7.0,100.646,1.0,SOFT,7.0,True,31.833,44.040,24.773,283.0,310.0,1,27.1,30.9,22.0,False,0.6,False,100.646,0,101.013958,100.6050,0.0410
4,ALB,WILLIAMS,8.0,100.918,1.0,SOFT,8.0,True,31.830,44.297,24.791,283.0,312.0,1,27.1,30.9,21.0,False,0.9,False,100.918,0,101.347284,100.7040,0.2140


## 3 · Leakage and availability screen

Before engineering anything we classify **every raw column** into one of three roles.
This is the single most important step for model validity, so it is done explicitly and
auditable rather than by silent omission.

* **Predictor** — known *before* the lap is run, therefore legitimately usable.
* **Target** — the quantity to be predicted.
* **Excluded (leakage)** — only knowable *during or after* the lap in question.

The exclusions matter here:

| Column | Why excluded |
|---|---|
| `Sector1Time`, `Sector2Time`, `Sector3Time` | They sum **exactly** to `LapTime`. Task 4's correlation report already flagged this. Keeping them makes lap-time prediction a trivial identity. |
| `SpeedFL`, `SpeedST` | Same-lap telemetry — measured *while* the lap is being driven, so unavailable at prediction time. |
| `IsPersonalBest` | A deterministic function of `LapTime` versus the driver's earlier laps. |

`Driver`, `Team` and `Compound` are retained: they are known before the lap starts and are
encoded as features in §5.6.

In [3]:
evidence = build.leakage_evidence(build.load_clean_laps())
print("Evidence for the sector-time exclusion")
print(f"  max |LapTime - sum(sectors)| = {evidence['max_residual_s']:.4f} s")
print(f"  median                       = {evidence['median_residual_s']:.2e} s")
print("  -> sectors reconstruct LapTime almost exactly; they are leakage, not features.")
print(f"\nexcluded: {build.LEAKAGE_COLS}")

Evidence for the sector-time exclusion
  max |LapTime - sum(sectors)| = 6.5018 s
  median                       = 0.00e+00 s
  -> sectors reconstruct LapTime almost exactly; they are leakage, not features.

excluded: ['Sector1Time', 'Sector2Time', 'Sector3Time', 'SpeedFL', 'SpeedST', 'IsPersonalBest']


## 4 · Target definition

Three targets are derived, covering the modelling needs of Tasks 6–8. They are stored in the
export so downstream tasks select rather than recompute.

1. **`target_laptime`** — lap time in seconds. *Primary regression target*; drives the feature
   selection funnel below.
2. **`target_pit_next_lap`** — binary: does this driver pit at the end of this lap? Derived
   from the stint index incrementing on the following lap. Supports the strategy-decision
   classifier, and connects directly to the pit-stop rules of the Task 2 expert system.
3. **`target_laptime_fuel_corrected`** — lap time with the fuel-burn trend removed, isolating
   *tyre* degradation from the car simply getting lighter. This is the quantity Task 3's
   search cost model reasons about.

The fuel-burn coefficient is estimated from the data (regressing lap time on lap number while
controlling for tyre life and compound) rather than hard-coded, so it stays correct if the
notebook is pointed at a real FastF1 session.

In [4]:
print(f"fuel-burn effect : {result['fuel_coef']:+.4f} s per lap (negative = car speeds up as fuel burns off)")
print(f"pit events       : {int(laps['target_pit_next_lap'].sum())} of {len(laps)} laps "
      f"({laps['target_pit_next_lap'].mean():.1%})")
laps[build.TARGETS].describe().round(3)

fuel-burn effect : -0.0613 s per lap (negative = car speeds up as fuel burns off)
pit events       : 48 of 995 laps (4.8%)


,target_laptime,target_pit_next_lap,target_laptime_fuel_corrected
count,995.000,995.000,995.000
mean,99.097,0.048,100.843
std,2.012,0.214,1.819
min,94.003,0.000,97.376
25%,97.706,0.000,99.727
50%,98.482,0.000,100.324
75%,99.892,0.000,101.280
max,103.985,1.000,107.296


## 5 · Feature engineering

Six thematic blocks. Two rules govern all of them:

**(a) Causality.** Anything derived from race history uses `shift(1)` — only laps strictly
before the current one.

**(b) No exact linear identities.** If a feature is an exact linear combination of others
(e.g. `TyreLife` = the sum of its per-compound interactions, or `TrackTemp` = `AirTemp` +
their delta), the redundant member is *not constructed*. Otherwise the VIF step in §6.3
would be forced to choose arbitrarily among perfectly collinear columns and could delete the
physically meaningful variable rather than the derived one.

We also express pace in **gap-space** — each driver's lap time relative to the field median on
that lap — which separates "how fast is the track right now" from "how fast is this driver
relative to everyone else", instead of entangling them in one correlated block.

The six blocks are `engineer_features` in `build.py`; each feature is registered with the
reason it exists, which is what ends up in `feature_provenance` in the metadata.

In [5]:
prov = meta["feature_provenance"]
pd.DataFrame({"feature": list(prov), "why it exists": list(prov.values())}).head(12)

,feature,why it exists
0,compound_medium,Compound contrast vs the HARD reference level.
1,compound_soft,Compound contrast vs the HARD reference level.
2,driver_alo,Driver identity (skill / style).
3,driver_bot,Driver identity (skill / style).
4,driver_dev,Driver identity (skill / style).
5,driver_gas,Driver identity (skill / style).
6,driver_ham,Driver identity (skill / style).
7,driver_hul,Driver identity (skill / style).
8,driver_lec,Driver identity (skill / style).
9,driver_mag,Driver identity (skill / style).


In [6]:
c = meta["preprocessing_contract"]
print(f"warm-up rows dropped : {c['warmup_rows_dropped']} (history features undefined there)")
print(f"first usable lap     : {c['first_usable_lap']}")
print(f"modelling rows       : {len(F)}  |  candidate features: {F.shape[1]}")

warm-up rows dropped : 60 (history features undefined there)
first usable lap     : 4
modelling rows       : 995  |  candidate features: 50


## 6 · Feature selection

A four-stage funnel, applied against the primary regression target `target_laptime`. Each
stage removes a *different* kind of unwanted feature, so the order matters:

| Stage | Removes | Rationale |
|---|---|---|
| 6.1 Near-zero variance | Constant / near-constant columns | Cannot inform any model |
| 6.2 Pairwise correlation | One of each redundant pair (\|r\| > 0.95) | Keeps the more informative twin |
| 6.3 Variance inflation | Multi-way collinearity (VIF > 10) | Pairwise checks miss 3-way redundancy |
| 6.4 Importance + stability | Weak / unstable predictors | Consensus across three different criteria |

Finally §6.5 chooses how many features to keep by measuring cross-validated error, and §6.6
verifies the reduced set against the full set.

### 6.0 · Validation strategy

Because this is a **time-ordered panel**, a random K-fold split would train on lap 50 and test
on lap 10 — leaking the future into the past and flattering every score.

Instead we use an **expanding-window, lap-forward split**: train on laps 1..k, test on the
next block of laps, then expand. Whole laps are kept together in a single fold so that the
field-median features cannot leak across the boundary. This mirrors how the model would
actually be used — predicting the remainder of a race from what has happened so far.

In [7]:
for i, (tr, te) in enumerate(result["splits"], 1):
    print(f"fold {i}: train {len(tr):>3} rows (laps <= {laps['LapNumber'].iloc[tr].max():>2})"
          f"  ->  test {len(te):>3} rows "
          f"(laps {laps['LapNumber'].iloc[te].min()}-{laps['LapNumber'].iloc[te].max()})")

fold 1: train 409 rows (laps <= 24.0)  ->  test 152 rows (laps 25.0-32.0)
fold 2: train 561 rows (laps <= 32.0)  ->  test 151 rows (laps 33.0-40.0)
fold 3: train 712 rows (laps <= 40.0)  ->  test 137 rows (laps 41.0-48.0)
fold 4: train 849 rows (laps <= 48.0)  ->  test 146 rows (laps 49.0-57.0)


In [8]:
funnel = meta["selection_funnel"]
print(f"1. near-zero variance dropped ({len(funnel['1_near_zero_variance_dropped'])}): "
      f"{funnel['1_near_zero_variance_dropped']}")
print(f"2. correlation dropped ({len(funnel['2_correlation_dropped'])}): {funnel['2_correlation_dropped']}")
print(f"3. VIF dropped ({len(funnel['3_vif_dropped'])}): {funnel['3_vif_dropped']}")
print(f"4. {funnel['4_importance_and_stability']}")
pd.DataFrame(result["correlation_pairs"]) if result["correlation_pairs"] else "no pairs above |r| 0.95"

1. near-zero variance dropped (1): ['is_rainfall']
2. correlation dropped (3): ['tyre_life_sq', 'air_temp', 'track_air_delta']
3. VIF dropped (1): ['race_progress']
4. ranked by mutual information + tree importance + L1 coefficient; K*=45 chosen within 2% of best CV MAE


,kept,dropped,|r|
0,tyre_life,tyre_life_sq,0.9561
1,race_progress,air_temp,0.9803
2,race_progress,track_air_delta,0.9718


In [9]:
result["ranking"].head(15).round(4)

,mutual_info,tree_importance,l1_coef,avg_rank,stability
field_median_lag1,0.5212,0.1472,0.4833,2.3333,1.00
stint_number,0.3841,0.0858,0.5276,3.3333,1.00
gap_expanding,0.4674,0.0575,0.4440,5.0000,1.00
track_status,0.0626,0.1291,0.4886,6.3333,0.50
tracktemp_dev_x_tyrelife,0.5982,0.0863,0.0224,6.3333,1.00
form_vs_baseline,0.2307,0.0405,0.4443,6.3333,1.00
field_pace_trend,0.2382,0.0173,0.2100,7.0000,1.00
gap_roll3_std,0.1904,0.0312,0.1030,8.6667,1.00
wind_speed,0.1954,0.0030,0.1650,10.0000,0.75
tyrelife_x_soft,0.1109,0.0099,0.0824,10.6667,1.00


In [10]:
mae_sel, r2_sel = result["cv_selected"]
mae_all, r2_all = result["cv_all_features"]
print(result["regression_sweep"].to_string())
print(f"\nselected K* = {len(meta['selected_features']['target_laptime'])}")
pd.DataFrame({
    "n_features": [len(meta["selected_features"]["target_laptime"]), result["features_post_funnel"].shape[1]],
    "cv_MAE_s": [round(mae_sel, 4), round(mae_all, 4)],
    "cv_R2": [round(r2_sel, 4), round(r2_all, 4)],
}, index=["selected", "all (post-funnel)"])

    cv_MAE_s   cv_R2
K                   
4     1.6486 -0.4631
6     1.2579  0.2217
8     1.2189  0.2648
10    1.2323  0.2576
12    1.0457  0.4077
15    1.0338  0.4237
20    1.0288  0.4377
45    0.9891  0.4491

selected K* = 45


,n_features,cv_MAE_s,cv_R2
selected,45,0.9891,0.4491
all (post-funnel),45,1.0001,0.4444


### Feature selection for the pit-decision target

Ranked the same way, but against the binary target and with `K` fixed at 8: with roughly 5%
positives, sweeping `K` would be choosing between numbers that are mostly noise. The CV AUC
by `K` is printed so that claim is checkable.

In [11]:
print("cv AUC by K: " + ", ".join(f"K={k}: {v:.3f}" for k, v in result["classification_sweep"].items()))
print(f"\nselected ({len(meta['selected_features']['target_pit_next_lap'])}): "
      f"{meta['selected_features']['target_pit_next_lap']}")
result["ranking_classification"].head(10).round(4)

cv AUC by K: K=4: 0.879, K=6: 0.868, K=8: 0.897, K=10: 0.902

selected (8): ['tyre_life', 'tracktemp_dev_x_tyrelife', 'form_vs_baseline', 'field_median_lag1', 'field_pace_trend', 'tyrelife_x_soft', 'gap_roll3_mean', 'compound_soft']


,mutual_info,tree_importance,l1_coef,avg_rank
tyre_life,0.0343,0.1687,3.5151,1.6667
tracktemp_dev_x_tyrelife,0.0519,0.1030,3.3245,2.0000
form_vs_baseline,0.0250,0.1091,0.4863,5.3333
field_median_lag1,0.0161,0.0818,0.9963,5.3333
field_pace_trend,0.0398,0.0578,0.5783,6.0000
tyrelife_x_soft,0.0142,0.0523,2.5729,6.3333
gap_roll3_mean,0.0043,0.0782,0.9856,10.3333
compound_soft,0.0057,0.0088,1.7775,12.0000
gap_roll3_std,0.0058,0.0680,0.2422,13.0000
team_alpine,0.0055,0.0081,0.3189,15.3333


## 7 · Export

Two files, and nothing else.

**`f1_features_selected.csv`** — identifier columns, the union of both selected feature sets,
and all three targets. The union is exported (rather than two separate files) so Tasks 6–8
read one dataset and pick columns via the metadata.

**`feature_metadata.json`** — the contract for downstream tasks: which columns are features
for which target, what each feature means and why it exists, which columns need scaling, and
the validation strategy that must be reused to keep results comparable.

A note on what is deliberately *not* exported: the features are **unscaled**. Standardising
here would compute means and standard deviations over the whole dataset, leaking test-fold
statistics into training. Tasks 6–8 should fit the scaler inside their own CV pipeline —
`numeric_features_requiring_scaling` in the metadata lists exactly which columns need it.

In [12]:
print(meta["validation_scores"]["caveat"])
print(f"\nwrote {Path(result['csv_path']).relative_to(ROOT)}")
print(f"wrote {Path(result['json_path']).relative_to(ROOT)}")
result["export"].head()

48 pit event(s) observed across 21 distinct laps (the busiest lap accounts for 12% of all pits) - a reasonably realistic spread of strategic pit timing. This remains a single session, though, and results should not be generalised beyond it.

wrote data/processed/f1_features_selected.csv
wrote data/processed/feature_metadata.json


,Driver,LapNumber,Stint,Compound,Team,compound_medium,compound_soft,driver_alo,driver_bot,driver_dev,driver_gas,driver_ham,driver_hul,driver_lec,driver_mag,driver_nor,driver_oco,driver_per,driver_pia,driver_rus,driver_sai,driver_sar,driver_str,driver_tsu,driver_ver,driver_zho,field_median_lag1,field_pace_trend,form_vs_baseline,gap_expanding,gap_roll3_mean,gap_roll3_std,humidity,is_fresh_tyre,stint_number,team_alphatauri,team_alpine,team_aston_martin,team_ferrari,team_haas_f1_team,team_mclaren,team_mercedes,team_red_bull_racing,team_williams,track_status,tracktemp_dev_x_tyrelife,tyre_life,tyrelife_x_medium,tyrelife_x_soft,wind_speed,target_laptime,target_pit_next_lap,target_laptime_fuel_corrected
0,ALB,4.0,1.0,SOFT,WILLIAMS,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,99.8625,-0.5565,0.183333,0.097167,0.097167,0.158867,22.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,5.162844,4.0,0.0,4.0,0.4,99.761,0,99.944979
1,ALB,5.0,1.0,SOFT,WILLIAMS,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,99.7920,-0.0705,-0.096125,0.065125,0.086833,0.169030,22.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,5.453555,5.0,0.0,5.0,0.9,100.606,0,100.851305
2,ALB,6.0,1.0,SOFT,WILLIAMS,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,100.6180,0.8260,-0.061700,0.049700,0.079167,0.174618,22.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,6.544265,6.0,0.0,6.0,0.6,100.744,0,101.050632
3,ALB,7.0,1.0,SOFT,WILLIAMS,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,100.5375,-0.0805,0.130667,0.075833,0.054500,0.131978,22.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,6.934976,7.0,0.0,7.0,0.6,100.646,0,101.013958
4,ALB,8.0,1.0,SOFT,WILLIAMS,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,100.6050,0.0675,-0.029857,0.070857,0.078500,0.113975,21.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,7.925687,8.0,0.0,8.0,0.9,100.918,0,101.347284


## 8 · Verification

Read both files back from disk and check the contract holds.

In [13]:
check_df = pd.read_csv(result["csv_path"])
check_meta = json.loads(Path(result["json_path"]).read_text())

feat_reg = check_meta["selected_features"]["target_laptime"]
assert list(check_df.columns) == list(result["export"].columns)
assert check_df[feat_reg].notna().all().all()
assert len(check_df) == check_meta["rows"]
for leak in build.LEAKAGE_COLS:
    assert leak not in check_df.columns

print("round-trip verification")
print(f"  rows / columns       : {check_df.shape[0]} x {check_df.shape[1]}")
print(f"  regression features  : {len(feat_reg)}")
print(f"  classification feats : {len(check_meta['selected_features']['target_pit_next_lap'])}")
print(f"  needs scaling        : {len(check_meta['numeric_features_requiring_scaling'])}")
print(f"  targets              : {list(check_meta['targets'])}")
print("\nTask 5 complete - ready for Task 6 (Classical Machine Learning).")

round-trip verification
  rows / columns       : 995 x 53
  regression features  : 45
  classification feats : 8
  needs scaling        : 14
  targets              : ['target_laptime', 'target_pit_next_lap', 'target_laptime_fuel_corrected']

Task 5 complete - ready for Task 6 (Classical Machine Learning).


## 9 · Hand-off to Task 6

**What this stage did**

1. Read Task 4's cleaned per-lap dataset in place — no dataset was copied or duplicated.
2. Excluded six columns as leakage, with the sector-time identity demonstrated numerically
   rather than asserted.
3. Engineered features across six blocks — tyre/stint dynamics, fuel and race progress, causal
   pace history in gap-space, field-level pace, environment, and reference-encoded
   categoricals — on a basis chosen to contain no exact linear identities.
4. Derived three targets covering regression, classification and fuel-corrected pace.
5. Ran a four-stage selection funnel (near-zero variance → correlation → VIF → importance with
   fold stability) and chose the feature count by cross-validated error under a parsimony rule.
6. Exported one dataset and one metadata file.

**How later tasks consume it**

```python
import json
import pandas as pd

df = pd.read_csv("data/processed/f1_features_selected.csv")
meta = json.loads(open("data/processed/feature_metadata.json").read())

X = df[meta["selected_features"]["target_laptime"]]
y = df["target_laptime"]
# Fit the scaler inside each CV fold - see meta["preprocessing_contract"].
```

In practice Task 6 reads this through `app.intelligence.features.contract`, which validates the
two files against each other before any model is trained.

**Rebuilding without this notebook**

```bash
python scripts/build_features.py     # same functions, same outputs
python scripts/build_all.py --force  # Tasks 1-8, including this stage
```